In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import tensorflow as tf
import numpy as np

In [2]:
# Load dataset
df = pd.read_csv("../data/heart_disease_data.csv")
total_missing = df.isna().sum().sum()
print("Total missing values:", total_missing)

Total missing values: 0


In [3]:
X = df.drop(columns=["target"])
y = df["target"].values

In [4]:
# Process dataset
cat_cols = ["cp", "restecg", "slope", "thal"]
num_cols = ["age", "trestbps", "chol", "thalach", "oldpeak", "ca"]
binary_cols = ["sex", "fbs", "exang"]

# OneHotEncoding for categorical attributes
encoder = OneHotEncoder(sparse_output=False)
X_cat_ohe = encoder.fit_transform(X[cat_cols])

# StandardScaler for numerical attributes
scaler = StandardScaler()
X_num_scaled = scaler.fit_transform(X[num_cols])

# Binary attributes
X_binary = X[binary_cols].values  

X_processed = np.hstack([X_cat_ohe, X_num_scaled, X_binary])
print("X shape before Processing:", X.shape)
print("X shape after Processing:", X_processed.shape)
print("First Row:\n", X_processed[0])


X shape before Processing: (606, 13)
X shape after Processing: (606, 23)
First Row:
 [ 1.          0.          0.          0.          1.          0.
  0.          0.          0.          1.          0.          0.
  0.          1.         -0.7021358  -0.09273778  0.18815239  0.01544279
 -0.89686172  1.24459328  1.          1.          1.        ]


In [5]:
# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, random_state=42, stratify=y
)

In [6]:
# Define model architecture
model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(X_train.shape[1],)),
        tf.keras.layers.Dense(8, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ]
)

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

In [7]:
history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    verbose=1
)

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print("Test accuracy:", test_acc)

Epoch 1/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5640 - loss: 0.7133  
Epoch 2/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5868 - loss: 0.6760 
Epoch 3/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6116 - loss: 0.6433 
Epoch 4/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6405 - loss: 0.6161 
Epoch 5/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6818 - loss: 0.5936 
Epoch 6/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7004 - loss: 0.5723 
Epoch 7/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7335 - loss: 0.5517 
Epoch 8/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7562 - loss: 0.5315 
Epoch 9/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7769 - loss: 0.5147 
Epoch 10/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7789 - loss: 0.4976 
Epoch 11/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7893 - loss: 0.4819 
Epoch 12/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy:

In [8]:
# Save model
model.save("heart_nn_py.keras")

In [9]:
# Load model
loaded_model = tf.keras.models.load_model('heart_nn_py.keras')
loaded_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 8)              │           192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 605 (2.37 KB)

 Trainable params: 201 (804.00 B)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 404 (1.58 KB)